In [ ]:
import os
import requests
import sounddevice as sd
import numpy as np
import tempfile
import scipy.io.wavfile as wav
from dotenv import load_dotenv
import time

# -------------------------
# Load environment variables
# -------------------------
load_dotenv()

HF_API_KEY = os.getenv("HF_API_KEY")
HF_STT_MODEL = os.getenv("HF_STT_MODEL", "openai/whisper-small")
HF_LLM_MODEL = os.getenv("HF_LLM_MODEL", "HuggingFaceH4/zephyr-7b-beta")
HF_TTS_MODEL = os.getenv("HF_TTS_MODEL", "facebook/fastspeech2-en-ljspeech")

HEADERS = {"Authorization": f"Bearer {HF_API_KEY}"}

# -------------------------
# RECORD AUDIO
# -------------------------
def record_audio(duration=5, fs=16000):
    print("🎤 Speak now...")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='int16')
    sd.wait()
    return fs, audio

# -------------------------
# STT: Hugging Face Whisper
# -------------------------
def transcribe(audio):
    fs, data = audio
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        wav.write(f.name, fs, data)
        f.flush()

        url = f"https://api-inference.huggingface.co/models/{HF_STT_MODEL}"
        with open(f.name, "rb") as audio_file:
            response = requests.post(url, headers=HEADERS, data=audio_file)

        result = response.json()
        text = result.get("text", "")
        return text

# -------------------------
# LLM: Hugging Face
# -------------------------
def generate_response(prompt):
    url = f"https://api-inference.huggingface.co/models/{HF_LLM_MODEL}"
    payload = {
        "inputs": f"<|user|>\n{prompt}\n<|assistant|>",
        "parameters": {"max_new_tokens": 200, "temperature": 0.7}
    }
    response = requests.post(url, headers=HEADERS, json=payload)
    try:
        return response.json()[0]["generated_text"]
    except (KeyError, IndexError):
        return "Sorry, I couldn't generate a response."

# -------------------------
# TTS: Hugging Face FastSpeech2
# -------------------------
def speak(text):
    url = f"https://api-inference.huggingface.co/models/{HF_TTS_MODEL}"
    response = requests.post(url, headers=HEADERS, json={"inputs": text})

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(response.content)
        f.flush()

        fs, data = wav.read(f.name)
        sd.play(data, fs)
        sd.wait()

# -------------------------
# MAIN LOOP
# -------------------------
def run():
    print("🔊 Voice Assistant Started. Press Ctrl+C to exit.")
    while True:
        try:
            audio = record_audio(duration=5)
            text = transcribe(audio)
            if not text.strip():
                continue

            print(f"User: {text}")
            response = generate_response(text)
            print(f"Assistant: {response}")
            speak(response)

            time.sleep(0.5)  # small pause to avoid overlapping audio
        except KeyboardInterrupt:
            print("\nExiting...")
            break
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(1)

# -------------------------
# ENTRYPOINT
# -------------------------
if __name__ == "__main__":
    run()


🔊 Voice Assistant Started. Press Ctrl+C to exit.
🎤 Speak now...
Error: ('Connection broken: IncompleteRead(15772 bytes read, 8766 more expected)', IncompleteRead(15772 bytes read, 8766 more expected))
🎤 Speak now...
Error: ('Connection broken: IncompleteRead(8313 bytes read, 16225 more expected)', IncompleteRead(8313 bytes read, 16225 more expected))
🎤 Speak now...
Error: ('Connection broken: IncompleteRead(15773 bytes read, 8765 more expected)', IncompleteRead(15773 bytes read, 8765 more expected))
🎤 Speak now...
Error: ('Connection broken: IncompleteRead(15773 bytes read, 8765 more expected)', IncompleteRead(15773 bytes read, 8765 more expected))
🎤 Speak now...
Error: ('Connection broken: IncompleteRead(15773 bytes read, 8765 more expected)', IncompleteRead(15773 bytes read, 8765 more expected))
🎤 Speak now...
Error: ('Connection broken: IncompleteRead(15773 bytes read, 8765 more expected)', IncompleteRead(15773 bytes read, 8765 more expected))
🎤 Speak now...

Exiting...
